In [1]:
from dotenv import load_dotenv 
from openai import OpenAI 

load_dotenv() 
openai_client = OpenAI() 

In [ ]:
from rag_helper import RAGBase 
from ingest import load_faq_data, build_index 

documents = load_faq_data()
index = build_index(documents)

In [7]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [8]:
assistant.rag("How do I run Olama locally?")

'I don’t see any information in the context about running **Olama locally**.\n\nThe only related local-run command in the FAQ is for **MCP Inspector**:\n\n```bash\nnpx @modelcontextprotocol/inspector\n```\n\nIf you meant a different tool, let me know the exact name and I can check the context again.'

### Function Calling 

In [11]:
def search(query): 
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"} 

    return index.search(
        query, 
        num_results=5,
        boost_dict=boost_dict, 
        filter_dict=filter_dict 
    )


# The JSON the llm will see, describing the function: 
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join discovered the course"}', call_id='call_erNzLoknYqjGpx43gqFjxEgC', name='search', type='function_call', id='fc_0f18d1af59339009006a2fece8ccf8819d8290a0f08c53e404', namespace=None, status='completed')]

In [ ]:
import json 
call = response.output[0] 
args = json.loads(call.arguments)    # seriealize function call arguments 

results = search(**args) # call the function
result_json = json.dumps(results, indent=2) # the RAG search results 

In [ ]:
# Extend message history -> llm is stateless 
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,   # links result to function call; useful for multiple function calls in one turn 
    "output": result_json,
})

# Ask the llm again -> Now it should have the right context and give the answer 
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text


In [20]:
# Usage cost 
usage = response.usage
usage.input_tokens, usage.output_tokens

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


### The Agentic Loop 

Three components of the agent: 
* instructions (gives role and behavior)
* tools 
* memory (message history)

In [23]:
# Instructions (developer prompt)
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [24]:
# function-call helper 
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
# Processing one response
question = "I just discovered the course. Can I join?"

messages = [
    {"role": "developer", "content": instructions}, 
    {"role": "user", "content": question}, 
]

response = openai_client.response.create(
    model="gpt-5.4.mini", 
    input=messages, 
    tools=[search_tool], 
)

messages.extend(response.output) 
has_function_calls = False # -> this will flag whether the model needs another API call 

for item in response.output: 
    if item.type == "function_call": 
        print("function_call:", item.name, item.arguments) 
        call_output = make_call(item) 
        messages.append(call_output)
        has_function_calls = True     

    elif item.type == "message": 
        print("ASSISTANT:")
        print(item.content[0].text)

#### Full Agentic Loop 

In [ ]:
it = 1 

while True: 
    print(f"Iteration #{it}...")
    has_function_calls = False 

    response = openai_client.response.create(
        model="gpt-5.4-mini", 
        input=messages, 
        tools=[search_tool], 
    )

    messages.extend(response.output)

    for item in response.output: 
        if item.type == "function_call": 
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item) 
            messages.append(call_output) 
            has_function_calls = True 

        elif item.type == "message": 
            print("ASSISTANT:") 
            print(item.content[0].text)
    
    it += 1 
    if has_function_calls == False: 
        break 

In [32]:
# In a function 
def agent_loop(
    instructions, question, model="gpt-5.4-mini"
) -> str: 
    messages = [
        {"role": "developer", "content": instructions}, 
        {"role": "user", "content": question},
    ]

    it = 1 
    while True: 
        print(f"Iter #{it}...")
        has_function_calls = False 

        response = openai_client.responses.create(
            model=model, 
            input=messages, 
            tools=[search_tool], 
        )

        messages.extend(response.output) 

        for item in response.output: 
            if item.type == "function_call": 
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item) 
                messages.append(call_output) 
                has_function_calls = True 

            elif item.type == "message": 
                print("ASSISTANT:")
                last_answer = item.content[0].text 
                print(item.content[0].text)
        
        it += 1 
        if has_function_calls == False: 
            break 

    return last_answer

In [33]:
agent_loop(instructions, "How do I run Olama locally?")

Iter #1...
function_call: search {"query":"Olama locally run Ollama local install start server FAQ"}
function_call: search {"query":"Ollama local setup run models on your machine FAQ"}
Iter #2...
ASSISTANT:
To run Ollama locally:

1. Install Ollama from: https://ollama.com/download
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model locally:
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. Test that the local Ollama server is running:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response that includes the available models.

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(res

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. Test that the local Ollama server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response that includes the available models.\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you run into a connection issue, you can also start the server directly with:\n```bash\nollama serve\n```\n\nWould you li

### Prompt Engineering 

In [ ]:
# Encouraging multiple searches 
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

In [35]:
# Restricting off-topic questions 
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

Iter #1...
function_call: search {"query":"queen gambit chess opening what is queen gambit"}
Iter #2...
function_call: search {"query":"queen's gambit chess opening FAQ course"}
Iter #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” so it looks like this is off-topic for the course.

If you meant something course-related, feel free to rephrase it. Otherwise, I can help with other course questions—would you like to explore anything else?


'I couldn’t find any course FAQ entry about “queen’s gambit,” so it looks like this is off-topic for the course.\n\nIf you meant something course-related, feel free to rephrase it. Otherwise, I can help with other course questions—would you like to explore anything else?'

#### Frameworks 

In [36]:
from toyaikit.llm import OpenAIClient 
from toyaikit.tools import Tools 
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [38]:
# Register with from schema (the json)
agent_tools = Tools() 
agent_tools.add_tool(search, search_tool)

In [40]:
# Register without schema (from python function)
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [41]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [43]:
chat_interface = IPythonChatInterface() # handles display in notebook
callback = DisplayingRunnerCallback(chat_interface) # renders model messages and tool calls as they happen 

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [44]:
result = runner.loop(
    prompt="How do I run Olama locally?", 
    callback=callback, 
)

-> Response received


-> Response received


In [46]:
result.cost # Check call cost 

CostInfo(input_cost=Decimal('0.0010995'), output_cost=Decimal('0.00117'), total_cost=Decimal('0.0022695'))

In [47]:
result.all_messages # full message history 

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama local run Ollama locally"}', call_id='call_fPhdTj5oCCiC1

In [48]:
# Continue conversation 
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
### Interactive chat-like workflow: 
runner.run() 